# PySpark: RDD vs DataFrame

---

## 1. Introducción

Apache Spark es un motor de procesamiento distribuido diseñado para trabajar con grandes volúmenes de datos de forma eficiente.

Dentro de Spark existen distintas abstracciones para manejar datos:

- RDD (Resilient Distributed Dataset) → Bajo nivel (fundacional)
- DataFrame → Alto nivel (estándar actual)
- Dataset → Tipado (principalmente en Scala/Java)

Este documento explica en profundidad:
- Arquitectura
- Funcionamiento interno
- Ejecución
- Optimización
- Casos reales
- Comparación técnica detallada

---

## 2. Arquitectura de Spark

### 2.1 Componentes principales

#### Driver
- Define el programa
- Construye el DAG
- Coordina ejecución

#### Cluster Manager
- Gestiona recursos
- Ejemplos:
  - Standalone
  - YARN
  - Kubernetes

#### Executors
- Ejecutan tareas
- Procesan particiones
- Mantienen datos en memoria

---

### 2.2 Flujo de ejecución

1. Usuario define operaciones  
2. Spark construye DAG  
3. Se optimiza el plan  
4. Se divide en stages  
5. Se ejecuta en paralelo  

---

## 3. RDD (Resilient Distributed Dataset)

### 3.1 Definición

Un RDD es una colección distribuida, inmutable y tolerante a fallos que permite procesamiento paralelo.

---

### 3.2 Características fundamentales

#### Inmutabilidad
- No se modifica
- Cada operación genera un nuevo RDD

#### Lazy Evaluation

```python
rdd2 = rdd.map(lambda x: x * 2)  # No ejecuta
rdd2.collect()  # Ejecuta


# Optimización en Spark: Catalyst Optimizer en Profundidad Avanzada

---

## 1. Introducción

En Apache Spark, el uso de DataFrames y Spark SQL permite describir transformaciones de datos de forma declarativa. Esto habilita a Spark a aplicar optimizaciones automáticas antes de ejecutar cualquier operación.

El componente central de este proceso es el Catalyst Optimizer, encargado de transformar un plan lógico en un plan físico altamente eficiente.

Este documento profundiza en:
- Arquitectura interna del optimizador
- Tipos de reglas
- Estrategias de ejecución
- Costos en sistemas distribuidos
- Detalles internos que afectan performance real

---

## 2. Filosofía del optimizador

Spark separa completamente:

- Qué hacer (definido por el usuario)
- Cómo hacerlo (decidido por Spark)

Esto permite aplicar múltiples optimizaciones sin modificar el código original.

---

## 3. Pipeline completo de optimización

Spark transforma una consulta en múltiples etapas:

1. Parsed Logical Plan  
2. Analyzed Logical Plan  
3. Optimized Logical Plan  
4. Physical Plan  
5. Ejecución  

Cada etapa transforma un árbol de operaciones.

---

## 4. Representación interna: Árboles

Todos los planes en Spark son representados como árboles:

Ejemplo:

```text
Project [area]
  └── Filter (monto > 100)
        └── Relation [area, monto]

In [0]:
# Ejemplo de uso de DataFrame en PySpark

# Crear un DataFrame a partir de una lista de diccionarios
data = [
    {"nombre": "Ana", "edad": 23},
    {"nombre": "Luis", "edad": 31},
    {"nombre": "Marta", "edad": 29}
]
df = spark.createDataFrame(data)

# Mostrar el DataFrame
display(df)

# Filtrar filas donde la edad es mayor a 25
df_filtrado = df.filter(df.edad > 25)
display(df_filtrado)

# Agrupar por nombre y calcular la edad máxima
df_agrupado = df.groupBy("nombre").agg({"edad": "max"})
display(df_agrupado)

# Ejemplo de uso de DataFrame en PySpark (equivalente a lo explicado con RDD)

# Ejemplo de inmutabilidad: cada transformación crea un nuevo DataFrame, el original no cambia
df_filtrado = df.filter(df.edad > 25)  # DataFrame nuevo, df original sigue igual
df_mapeado = df.withColumn("edad_mas_1", df.edad + 1)  # Otro DataFrame nuevo

# Recoger los resultados filtrados
resultados_df = df_filtrado.collect()

# Mostrar los resultados filtrados
for row in resultados_df:
    print(f"Nombre: {row['nombre']}, Edad: {row['edad']}")

# Seleccionar solo los nombres
df_nombres = df.select("nombre")
print([row['nombre'] for row in df_nombres.collect()])

# flatMap equivalente: explode solo array de un solo tipo (ejemplo con 'nombre')
from pyspark.sql.functions import array, explode
df_flat = df.select(explode(array("nombre")).alias("elemento"))
print([row['elemento'] for row in df_flat.collect()])

# distinct: elimina duplicados
data_con_duplicados = [
    {"nombre": "Ana", "edad": 23},
    {"nombre": "Ana", "edad": 23},
    {"nombre": "Luis", "edad": 31}
]
df_con_duplicados = spark.createDataFrame(data_con_duplicados)
df_distinct = df_con_duplicados.distinct()
print(df_distinct.collect())

# union: une dos DataFrames
df_extra = spark.createDataFrame([{"nombre": "Pedro", "edad": 40}])
df_union = df.union(df_extra)
print(df_union.collect())

# Ejecución lazy: las transformaciones no se ejecutan hasta una acción (collect, count, etc.)
from pyspark.sql.functions import upper
df_lazy = df.withColumn("nombre_mayus", upper(df.nombre))
# Nada se ejecuta hasta que se llama a una acción:
print(df_lazy.collect())

# filter, select, withColumn, distinct, union son transformaciones (crean nuevos DataFrames)
# collect, count, take, first son acciones (disparan la ejecución)

# count: acción que cuenta los elementos
print(df.count())

# take: acción que devuelve los primeros n elementos
print(df.take(2))

# first: acción que devuelve el primer elemento
print(df.first())

In [0]:
df_con_duplicados.explain(True)

##Ejemplo de Explain samples.nyctaxi.trips

In [0]:
from pyspark.sql.functions import col, year, month, when

df_taxi = (
    spark.read.table("samples.nyctaxi.trips")
    .filter(col("trip_distance").between(2, 10))
    .filter(col("fare_amount") > 15)
    .filter(col("pickup_zip").isin("10001", "10002", "10003"))
    .select(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_distance",
        "fare_amount",
        "pickup_zip",
        "dropoff_zip"
    )
    .withColumn("pickup_year", year(col("tpep_pickup_datetime")))
    .withColumn("pickup_month", month(col("tpep_pickup_datetime")))
    .withColumn("trip_category", when(col("trip_distance") > 5, "Largo").otherwise("Corto"))
    .filter(col("pickup_year") == 2025)  
)

df_taxi.explain(True)


## Spark Shuffle Hash Join vs Broadcast Hash Join

### Shuffle Hash Join

- **¿Qué es?**  
  Un tipo de join donde Spark redistribuye (shuffles) los datos de ambos DataFrames por la clave de join.  
- **¿Cuándo se usa?**  
  Cuando ambos DataFrames son grandes y no caben en memoria de un solo executor.
- **Funcionamiento:**  
  Spark divide ambos DataFrames en particiones, las reorganiza para que las filas con la misma clave estén juntas, y luego aplica el join usando una tabla hash.
- **Ventaja:**  
  Escala para grandes volúmenes de datos.
- **Desventaja:**  
  El shuffle es costoso: implica mover datos por la red, lo que puede afectar el rendimiento.

### Broadcast Hash Join

- **¿Qué es?**  
  Un join optimizado donde Spark copia (broadcast) el DataFrame pequeño a todos los executors.
- **¿Cuándo se usa?**  
  Cuando uno de los DataFrames es pequeño (cabe en memoria).
- **Funcionamiento:**  
  Spark envía el DataFrame pequeño a todos los nodos, y cada executor realiza el join localmente usando una tabla hash.
- **Ventaja:**  
  Evita el shuffle, mejora el rendimiento y reduce el tráfico de red.
- **Desventaja:**  
  Solo funciona si el DataFrame pequeño cabe en memoria.

---

**Resumen:**  
- Shuffle Hash Join: ambos DataFrames grandes → shuffle de datos.
- Broadcast Hash Join: uno pequeño → broadcast, join local, más eficiente.

In [0]:
from pyspark.sql.functions import broadcast

# Dataset inventado: tabla de códigos postales con área
data_zip = [
    {"zip": "10001", "area": "Chelsea"},
    {"zip": "10002", "area": "Lower East Side"},
    {"zip": "10003", "area": "East Village"},
    {"zip": "10004", "area": "Financial District"}
]
df_zip = spark.createDataFrame(data_zip)

# Shuffle Hash Join: join normal (Spark decide el tipo, suele ser shuffle hash si ambos son grandes)
df_shuffle_join = df_taxi.join(df_zip, df_taxi.pickup_zip == df_zip.zip, "inner")
#df_shuffle_join.explain(True)

# Broadcast Join: forzar broadcast del dataset pequeño (df_zip)

df_broadcast_join = df_taxi.join(broadcast(df_zip), df_taxi.pickup_zip == df_zip.zip, "inner").explain(True)


In [0]:
df_broadcast_join.explain(True)

## Persistencia / Cache estratégico

Spark permite almacenar (cache/persist) los resultados intermedios de transformaciones en memoria o disco. Esto es útil cuando:

- Un DataFrame/RDD se reutiliza varias veces en el pipeline.
- Las operaciones son costosas y se repiten en diferentes ramas del DAG.

### ¿Cómo funciona?

- `cache()`: Guarda los datos en memoria (RAM).
- `persist()`: Permite elegir el nivel de almacenamiento (memoria, disco, ambos).

### Ventajas

- Reduce el tiempo de ejecución en workflows complejos.
- Evita recalcular datos intermedios.
- Mejora el rendimiento en análisis iterativos.

### Ejemplo

python
df_filtrado = df.filter(df.edad > 25).cache()
# Ahora df_filtrado se reutiliza sin recalcular


### Estrategia

- Cache solo los DataFrames/RDDs que se usan varias veces.
- Liberar cache con `unpersist()` cuando ya no se necesite.
- Monitorizar el uso de memoria para evitar saturar los recursos.

---

## UDF vs Pandas UDF (Vectorización)

### UDF (User Defined Function)

- Permite definir funciones personalizadas en Python para aplicar sobre columnas de DataFrame.
- Procesa cada fila de forma individual (row-by-row).
- No aprovecha la vectorización de operaciones.
- Menor rendimiento: cada fila se convierte entre JVM y Python, lo que genera overhead.

python
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

def sumar_1(x):
    return x + 1

udf_sumar_1 = udf(sumar_1, IntegerType())
df.withColumn("edad_mas_1", udf_sumar_1(df.edad))


---

### Pandas UDF (Vectorized UDF)

- Usa Apache Arrow para procesar datos en bloques (vectorizados).
- La función recibe y devuelve objetos pandas Series.
- Mucho más eficiente: reduce conversiones, aprovecha operaciones vectorizadas de pandas/numpy.
- Ideal para operaciones complejas o que requieren alto rendimiento.

python
from pyspark.sql.functions import pandas_udf

@pandas_udf("int")
def sumar_1_vectorizado(serie):
    return serie + 1

df.withColumn("edad_mas_1", sumar_1_vectorizado(df.edad))


---

**Resumen:**  
- UDF: fila a fila, lento, sin vectorización.
- Pandas UDF: bloque a bloque, rápido, vectorizado, recomendado para funciones personalizadas.


Apache Arrow es un formato de datos en memoria, columnar y altamente eficiente, diseñado para que distintos sistemas (como Apache Spark, Pandas, Python, R, etc.) puedan compartir datos sin tener que copiarlos o serializarlos constantemente.

Arrow permite pasar datos entre JVM (Spark) y Python rápido y sin overhead innecesario.

| Tipo        | Input       | Output      | Uso                  |
| ----------- | ----------- | ----------- | -------------------- |
| Scalar      | Series      | Series      | Transformaciones     |
| Grouped Map | DataFrame   | DataFrame   | Lógica por grupo     |
| Grouped Agg | Series      | Escalar     | Agregaciones         |
| Iterator    | Iterator    | Iterator    | Performance avanzada |
| mapInPandas | Iterator DF | Iterator DF | Flexible / moderno   |


In [0]:
# Ejemplo de UDF vs Pandas UDF en PySpark usando samples.nyctaxi.trips

from pyspark.sql.functions import udf, pandas_udf
from pyspark.sql.types import IntegerType

# Filtrar y seleccionar columnas relevantes del dataset de taxis
df_taxi = (
    spark.read.table("samples.nyctaxi.trips")
    .filter(col("trip_distance").between(2, 10))
    .filter(col("fare_amount") > 15)
    .filter(col("pickup_zip").isin("10001", "10002", "10003"))
    .select("tpep_pickup_datetime", "trip_distance", "fare_amount", "pickup_zip")
    .limit(100)
)

# UDF tradicional: suma 1 al trip_distance
def sumar_1(x):
    return x + 1

udf_sumar_1 = udf(sumar_1, IntegerType())
df_udf = df_taxi.withColumn("trip_distance_mas_1_udf", udf_sumar_1(df_taxi.trip_distance))
display(df_udf)

# Pandas UDF: suma 1 al trip_distance (vectorizado)
@pandas_udf("int")
def sumar_1_vectorizado(serie):
    return serie + 1

df_pandas_udf = df_taxi.withColumn("trip_distance_mas_1_pandas_udf", sumar_1_vectorizado(df_taxi.trip_distance))
display(df_pandas_udf)

In [0]:
df_pandas_udf.explain(True)